In [0]:
%run /Workspace/Users/chittoriarocks@gmail.com/Gold_files/framework/merge_helper

In [0]:
%run /Workspace/Users/chittoriarocks@gmail.com/Gold_files/framework/watermark_helper

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

table_name = "dim_customer"

print(f"Starting Gold Load for: {table_name}")

# =========================================================
# STEP 1 — GET WATERMARK
# =========================================================

watermark = get_watermark(table_name)

print(f"Watermark: {watermark}")

# =========================================================
# STEP 2 — READ CHANGED RECORDS ONLY
# =========================================================

customers_df = (
    spark.table("silver.customers")
    .filter(
        F.col("updated_at") > F.lit(watermark)
    )
)

addresses_df = spark.table("silver.addresses")

# =========================================================
# STEP 3 — BUILD GOLD DATASET
# =========================================================

dim_customer_df = (
    customers_df.alias("c")
    .join(
        addresses_df.alias("a"),
        on="customer_id",
        how="left"
    )
    .select(
        F.col("c.customer_id").alias("customer_id"),
        F.col("c.first_name").alias("first_name"),
        F.col("c.last_name").alias("last_name"),
        F.col("c.email").alias("email"),
        F.col("a.city").alias("city"),
        F.col("a.state").alias("state")
    )
    .withColumn(
        "gold_updated_ts",
        F.current_timestamp()
    )
)

# =========================================================
# STEP 4 — REMOVE DUPLICATES
# =========================================================

window_spec = Window.partitionBy(
    "customer_id"
).orderBy(
    F.col("gold_updated_ts").desc()
)

dim_customer_df = (
    dim_customer_df
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# =========================================================
# STEP 5 — CHECK EMPTY DATAFRAME
# =========================================================

if dim_customer_df.limit(1).count() == 0:

    print("No incremental data found")

else:

    print("Merging incremental records")

    # =====================================================
    # STEP 6 — MERGE INTO GOLD
    # =====================================================

    execute_merge(
        spark=spark,

        source_df=dim_customer_df,

        target_table="gold.dim_customer",

        merge_condition="""
            t.customer_id = s.customer_id
        """,

        update_map={
            "first_name": "s.first_name",
            "last_name": "s.last_name",
            "email": "s.email",
            "city": "s.city",
            "state": "s.state",
            "gold_updated_ts": "s.gold_updated_ts"
        },

        insert_map={
            "customer_id": "s.customer_id",
            "first_name": "s.first_name",
            "last_name": "s.last_name",
            "email": "s.email",
            "city": "s.city",
            "state": "s.state",
            "gold_updated_ts": "s.gold_updated_ts"
        }
    )

    # =====================================================
    # STEP 7 — UPDATE WATERMARK
    # =====================================================

    update_watermark(table_name)

    print("Gold Merge Completed")